In [2]:
from datasets import load_dataset
import numpy as np
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from torch.utils.data import BatchSampler, Sampler, DataLoader

In [4]:
ds = load_dataset("csv", data_files={"train": "train.csv", "test": "test.csv", "sample_submissions": "sample_submission.csv"}
)

In [5]:
ds.set_format("pandas")

In [6]:
goods_image_vectors = np.load("D:\jup\goods_image_vectors\goods_image_vectors\embed_deperson.npy")
goods_image_items = np.load("D:\jup\goods_image_vectors\goods_image_vectors\items_deperson.npy")
goods_tittle_vectors = np.load("D:\jup\goods_title_vectors\goods_title_vectors\embed_deperson.npy")
goods_tittle_items = np.load("D:\jup\goods_title_vectors\goods_title_vectors\items_deperson.npy")
offer_image_vectors = np.load("D:\jup\offer_image_vectors\offer_image_vectors\embed_deperson.npy")
offer_image_items = np.load("D:\jup\offer_image_vectors\offer_image_vectors\items_deperson.npy")
offer_tittle_vectors = np.load("D:\jup\offer_title_vectors\offer_title_vectors\embed_deperson.npy")
offer_tittle_items = np.load("D:\jup\offer_title_vectors\offer_title_vectors\items_deperson.npy")

print(len(goods_image_vectors), len(goods_image_items))

<>:1: SyntaxWarning: invalid escape sequence '\j'
<>:2: SyntaxWarning: invalid escape sequence '\j'
<>:3: SyntaxWarning: invalid escape sequence '\j'
<>:4: SyntaxWarning: invalid escape sequence '\j'
<>:5: SyntaxWarning: invalid escape sequence '\j'
<>:6: SyntaxWarning: invalid escape sequence '\j'
<>:7: SyntaxWarning: invalid escape sequence '\j'
<>:8: SyntaxWarning: invalid escape sequence '\j'
<>:1: SyntaxWarning: invalid escape sequence '\j'
<>:2: SyntaxWarning: invalid escape sequence '\j'
<>:3: SyntaxWarning: invalid escape sequence '\j'
<>:4: SyntaxWarning: invalid escape sequence '\j'
<>:5: SyntaxWarning: invalid escape sequence '\j'
<>:6: SyntaxWarning: invalid escape sequence '\j'
<>:7: SyntaxWarning: invalid escape sequence '\j'
<>:8: SyntaxWarning: invalid escape sequence '\j'
C:\Users\volko\AppData\Local\Temp\ipykernel_11912\1839093641.py:1: SyntaxWarning: invalid escape sequence '\j'
  goods_image_vectors = np.load("D:\jup\goods_image_vectors\goods_image_vectors\embed_dep

317707 317707


In [7]:
print(len(offer_image_vectors), len(offer_image_items))

457586 457586


In [8]:
goods_image_items = list(map(str, goods_image_items))
goods_tittle_items = list(map(str, goods_tittle_items))
offer_image_items = list(map(str, offer_image_items))
offer_tittle_items = list(map(str, offer_tittle_items))

In [9]:

goods_image = dict(zip(goods_image_items, torch.from_numpy(goods_image_vectors)))
goods_tittle = dict(zip(goods_tittle_items, torch.from_numpy(goods_tittle_vectors)))
offers_image = dict(zip(offer_image_items, torch.from_numpy(offer_image_vectors)))
offers_tittle = dict(zip(offer_tittle_items, torch.from_numpy(offer_tittle_vectors)))

In [10]:
sets  = (
    set(goods_image.keys()) & set(goods_tittle.keys()) & set(offers_image.keys()) 
    & set(offers_tittle.keys())
)

In [11]:
len(sets)

123515

In [12]:
goods_image_s, goods_tittle_s, offers_image_s, offers_tittle_s = {}, {}, {}, {}

for k in sets:
    goods_image_s[k] = goods_image[k]
    goods_tittle_s[k] = goods_tittle[k]
    offers_image_s[k] = offers_image[k]
    offers_tittle_s[k] = offers_tittle[k] 

In [13]:
len(goods_image_s)

123515

In [14]:
df = ds["train"].to_pandas()
dt = ds["test"].to_pandas()

In [15]:
df = df[df["target"]==1].dropna()

In [16]:
s = (df["goods_price"]- df["offer_price"]).abs().median()

In [17]:
df_wo_str = df.drop(columns="id")

In [18]:
df_wo_str.corr()

,offer_depersanalised,goods_depersanalised,sum_length,attrs+title_score,offer_price,goods_price,goods_category_id,target
offer_depersanalised,1.000000,-0.001288,0.084979,0.091408,0.073801,0.029088,-0.015824,NaN
goods_depersanalised,-0.001288,1.000000,-0.032544,0.080287,-0.012044,0.069469,-0.013550,NaN
sum_length,0.084979,-0.032544,1.000000,0.150886,0.088058,0.291534,0.137780,NaN
attrs+title_score,0.091408,0.080287,0.150886,1.000000,0.004807,0.008850,-0.019941,NaN
offer_price,0.073801,-0.012044,0.088058,0.004807,1.000000,0.068414,0.041237,NaN
goods_price,0.029088,0.069469,0.291534,0.008850,0.068414,1.000000,0.131366,NaN
goods_category_id,-0.015824,-0.013550,0.137780,-0.019941,0.041237,0.131366,1.000000,NaN
target,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df.head()

,offer_depersanalised,goods_depersanalised,sum_length,attrs+title_score,offer_price,goods_price,goods_category_id,target,id
254,118303,832186,54,0.262939,8520,8320.0,19.0,1,118303$832186
374,118088,832202,54,0.258545,6270,6120.0,19.0,1,118088$832202
413,118118,839035,54,0.251221,6790,6620.0,19.0,1,118118$839035
542,21953,843075,55,0.166870,131234,113099.0,8.0,1,21953$843075
1490,272114,860295,60,0.196411,112666,89455.0,8.0,1,272114$860295


In [20]:
dt["price_different"] = dt["goods_price"]-dt["offer_price"].abs()

In [21]:
dt.sort_values(by="price_different", ascending=False)

,offer_depersanalised,goods_depersanalised,sum_length,attrs+title_score,offer_price,goods_price,goods_category_id,target,id,price_different
271688,424740,41073,795,0.001225,6162,1.638016e+09,6.000000e+00,NaN,424740$41073,1.638010e+09
311160,423333,842619,1040,0.869141,22560,1.638016e+09,6.000000e+00,NaN,423333$842619,1.637994e+09
267874,423333,41038,776,0.000038,22560,1.638016e+09,6.000000e+00,NaN,423333$41038,1.637994e+09
303740,424712,1560422,983,0.864258,20690,1.560016e+09,6.000000e+00,NaN,424712$1560422,1.559995e+09
345304,421937,41163,1380,0.051178,15700,8.120822e+07,4.000000e+00,NaN,421937$41163,8.119252e+07
...,...,...,...,...,...,...,...,...,...,...
363807,24514,209319,3949,0.001567,2000,NaN,2.463685e+15,NaN,24514$209319,NaN
363825,314579,130243,4681,0.000061,4008,NaN,2.463685e+15,NaN,314579$130243,NaN
363827,208075,303414,4918,0.000053,678,NaN,2.463685e+15,NaN,208075$303414,NaN
363829,514821,130172,4955,0.000035,2238,NaN,2.463685e+15,NaN,514821$130172,NaN


In [22]:
print(type(new_df))

NameError: name 'new_df' is not defined

In [23]:
df['price_different'] = abs(df['offer_price'] - df['goods_price'])
new_df = df.drop(columns=['sum_length', 'offer_price', 'goods_price'])
new_df = new_df.rename(columns={'price_different':'price_difference'})

In [24]:
new_df.head()

,offer_depersanalised,goods_depersanalised,attrs+title_score,goods_category_id,target,id,price_difference
254,118303,832186,0.262939,19.0,1,118303$832186,200.0
374,118088,832202,0.258545,19.0,1,118088$832202,150.0
413,118118,839035,0.251221,19.0,1,118118$839035,170.0
542,21953,843075,0.166870,8.0,1,21953$843075,18135.0
1490,272114,860295,0.196411,8.0,1,272114$860295,23211.0


In [25]:
new_df['pair']= tuple(zip(df['offer_depersanalised'], df['goods_depersanalised'], df['target']))
new_df.head()

,offer_depersanalised,goods_depersanalised,attrs+title_score,goods_category_id,target,id,price_difference,pair
254,118303,832186,0.262939,19.0,1,118303$832186,200.0,"(118303, 832186, 1)"
374,118088,832202,0.258545,19.0,1,118088$832202,150.0,"(118088, 832202, 1)"
413,118118,839035,0.251221,19.0,1,118118$839035,170.0,"(118118, 839035, 1)"
542,21953,843075,0.166870,8.0,1,21953$843075,18135.0,"(21953, 843075, 1)"
1490,272114,860295,0.196411,8.0,1,272114$860295,23211.0,"(272114, 860295, 1)"


In [26]:
ks = goods_image_s.keys()

mask = [str(a[0]) in ks and str(a[1]) in ks for a in new_df['pair'].to_list()]
new_df = new_df[mask].reset_index(drop=True)
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25298 entries, 0 to 25297
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   offer_depersanalised  25298 non-null  int64  
 1   goods_depersanalised  25298 non-null  int64  
 2   attrs+title_score     25298 non-null  float64
 3   goods_category_id     25298 non-null  float64
 4   target                25298 non-null  int64  
 5   id                    25298 non-null  object 
 6   price_difference      25298 non-null  float64
 7   pair                  25298 non-null  object 
dtypes: float64(3), int64(3), object(2)
memory usage: 1.5+ MB


In [27]:
new_df = new_df.drop_duplicates(subset='pair', keep='first')

In [28]:
len(new_df['pair'])

25053

In [39]:
class data(Dataset):
    def __init__(self, table, img_g, img_o, tittle_g, tittle_o, pairs):
        self.table = table
        self.img_g = img_g
        self.img_o = img_o
        self.tittle_o = tittle_o
        self.tittle_g = tittle_g
        self.pairs = pairs

    def __getitem__(self, i):
        pair_ = self.pairs[i]
        id_o, id_g, y = pair_

        return {
            'img_goods':self.img_g[str(id_g)],
            'img_offers': self.img_o[str(id_o)],
            'tittle_goods': self.tittle_g[str(id_g)],
            'tittle_offers': self.tittle_o[str(id_o)],
            'difference': torch.tensor(self.table.loc[self.table['pair']== pair_, 'price_difference'].item(), dtype=torch.float32),
            'category': self.table.loc[self.table['pair']== pair_, 'goods_category_id'].item(),
            'y': y            
        }


    
    def __len__(self):
        return len(self.pairs)

        
        

In [30]:
import random

In [40]:
class SortByCategory(Sampler):
    def __init__(self, dataset, batch_size=32):
        self.dataset = dataset
        self.batch_size = batch_size

        self.ind_cat = {}
        for idx in range(len(dataset)):
            cat = dataset[idx]['category']
            if cat not in self.ind_cat:
                self.ind_cat[cat] = []
            self.ind_cat[cat].append(idx)

        self.batches = []
        for cat, ind in self.ind_cat.items():
            random.shuffle(ind)
            for i in range(0, len(ind), batch_size):
                self.batches.append(ind[i:i+batch_size])



        random.shuffle(self.batches)

        def __iter__(self):
            for batch in self.batches:
                yield batch

        def __len__(self):
            return len(self.batches)
        
    

In [135]:
dataset = data(new_df, goods_image_s, offers_image_s, goods_tittle_s, offers_tittle_s, new_df['pair'].to_list())

In [136]:
dataset.info()

AttributeError: 'data' object has no attribute 'info'

In [137]:
batch_sampler = SortByCategory(dataset=dataset, batch_size=32)

In [138]:
dataloader = DataLoader(dataset, batch_sampler=batch_sampler)

In [48]:
dataset[12]

{'img_goods': tensor([ 0.1014, -0.8819,  0.7784,  2.0945,  0.6039,  1.0421,  1.1884,  4.4960,
          0.8402,  0.4606,  1.0210, -0.9865,  0.3255,  3.1118,  1.4009,  0.9078,
         -1.1150,  0.2217, -3.3405, -2.5588,  1.9875, -0.4487, -1.7720,  2.8729,
          0.7391, -0.3400, -1.3706,  0.1450,  1.3011,  2.0324,  0.9299, -0.6019,
         -0.5901, -0.6243,  0.5309,  1.9642, -0.9100,  1.7781,  0.7793,  0.7115,
         -1.3458,  1.5033,  2.7636, -0.3496,  0.9722, -1.0996,  1.6362, -1.7350,
         -1.0631,  1.3505, -1.6814,  0.8571,  0.5529, -0.0457,  2.1859, -0.0758,
          0.2737,  0.1896,  2.8719, -1.7859, -0.9248,  2.4122,  0.5412,  0.9646,
          2.7235, -1.4182,  0.4996,  2.1340,  1.0561,  1.7337, -0.0592, -0.1529,
         -1.7685,  0.8766, -0.0453, -0.4567, -2.0834,  0.9449, -1.0032,  3.6631,
         -0.6336,  2.4191, -0.7582,  0.5572, -0.0563,  0.9878,  0.0133,  1.4373,
          2.0424, -0.9612, -1.7577,  0.9195,  1.9177,  1.9461, -0.5334,  1.2973,
          0.444

In [139]:
class myModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.Sequential(nn.Linear(256, 128), nn.LayerNorm(128), nn.ReLU())
        self.sc = nn.Sequential(nn.Linear(1, 16), nn.LayerNorm(16))
        self.f = nn.Sequential(nn.Linear(128, 64), nn.LayerNorm(128), nn.ReLU())
        self.to64 = nn.Linear(64*4+16, 64)
        self.relu = nn.ReLU()
        self.f = nn.Linear(128, 64)
        self.ln128 = nn.LayerNorm(128)
        self.layernorm64 = nn.LayerNorm(64)
        self.head = nn.Linear(64, 1)


    def forward(self, i_g, i_o, t_g, t_o, p_d):
        i_g_128 = self.p(i_g)
        i_o_128 = self.p(i_o)
        i_g_64 = self.f(i_g_128)
        i_o_64 = self.f(i_g_128)
        p_d_16 = self.sc(p_d.unsqueeze(0))
        t_g_64 = t_g
        t_o_64 = t_o
        x = torch.cat([i_g_64, i_o_64, t_g_64, t_o_64 ,p_d_16], dim=-1)
        x = self.to64(x)
        x = self.relu(x)
        x = self.layernorm64(x)
        itg = self.head(x)

        return itg.squeeze(-1)
        

In [140]:
model = myModel()

In [142]:
d = dataset[0]

In [143]:
print(d['tittle_goods'].dim())
print(d['img_goods'].dim())
print(d['difference'].dim())

1
1
0


In [128]:
len(d['img_goods'])

256

In [129]:
z = torch.cat([d['tittle_goods'], d['tittle_offers']], dim=-1)
z

tensor([ 0.0331, -0.0389, -0.0673,  0.0877, -0.1101,  0.0640,  0.0875,  0.0638,
        -0.0395,  0.0329, -0.0607, -0.0452,  0.0022, -0.0978,  0.0819, -0.1081,
         0.1948,  0.1021, -0.0332, -0.0078, -0.0195, -0.0535,  0.0536,  0.0163,
         0.0802,  0.1094, -0.0042, -0.0624,  0.0054, -0.0735, -0.1730, -0.0931,
        -0.0831,  0.0831, -0.0836,  0.0087,  0.0966, -0.0715, -0.1781, -0.0016,
         0.0426,  0.0326, -0.0892, -0.0171,  0.0079,  0.0879,  0.0014,  0.0949,
         0.0138,  0.0974,  0.0372, -0.0817,  0.0505, -0.1765, -0.1270, -0.0375,
        -0.0168,  0.0430,  0.0841, -0.1094, -0.0133, -0.0390,  0.0239,  0.1465,
         0.0168,  0.1666, -0.0356, -0.0580,  0.0037, -0.0450,  0.0459, -0.0033,
        -0.0759,  0.0774,  0.0372,  0.0220,  0.0288,  0.0289, -0.0124,  0.2080,
        -0.0809,  0.1547,  0.0771,  0.0277,  0.2278, -0.0572, -0.0476, -0.0018,
        -0.1110,  0.0115, -0.0196, -0.0759, -0.0135,  0.0615,  0.0905, -0.0010,
         0.0518,  0.0072,  0.0024,  0.06

In [144]:
logits = model(d['img_goods'], d['img_offers'], d['tittle_goods'], d['tittle_offers'], d['difference'])

In [145]:
logits

tensor(1.2361, grad_fn=<SqueezeBackward1>)